In [1]:
import tensorflow as tf

if tf.config.list_physical_devices('GPU'):
    print("GPU is available!")
    print(tf.config.list_physical_devices('GPU'))
else:
    print("GPU is not available. Please check runtime settings.")

GPU is not available. Please check runtime settings.


In [2]:
import os
import pandas as pd
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
from concurrent.futures import ThreadPoolExecutor
import joblib
from sklearn.utils import shuffle
from sklearn.model_selection import GridSearchCV

In [3]:
# Đường dẫn đến dữ liệu trên Kaggle
base_dir = '/kaggle/input/cs114-all-cars/'
csv_dir = '/kaggle/input/csv-slipts/'
save_dir = '/kaggle/working/'  # Thư mục lưu kết quả

# Thay đổi đoạn code tải model
weights_path = '/kaggle/input/pretrained-weights/mobilenet_v2_weights_tf_dim_ordering_tf_kernels_1.0_224_no_top.h5'
model = MobileNetV2(weights=weights_path, include_top=False, input_shape=(224, 224, 3))

# Bản đồ từ tên hiệu xe sang CategoryID
category_map = {
    "Others": 0,
    "Honda": 1,
    "Hyundai": 2,
    "KIA": 3,
    "Mazda": 4,
    "Mitsubishi": 5,
    "Suzuki": 6,
    "Toyota": 7,
    "VinFast": 8
}

In [4]:
# Chỉ xử lý 1 Split
split_index = 1

train_path = os.path.join(csv_dir, f"CarDataset-Splits-{split_index}-Train.csv")
test_path = os.path.join(csv_dir, f"CarDataset-Splits-{split_index}-Test.csv")

# Đọc dữ liệu Train và Test
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

In [5]:
# Trích xuất đặc trưng từ MobileNet
def extract_features(df, split_name):
    features_file = os.path.join(save_dir, f"Features_{split_name}_Split_{split_index}.npz")

    # Nếu tệp đã tồn tại, tải lại
    if os.path.exists(features_file):
        print(f"Loading features from {features_file}")
        data = np.load(features_file)
        return data['features'], data['labels']

    features, labels = [], []

    def process_image(row):
        try:
            image_path = os.path.join(base_dir, row['ImageFullPath'])
            label = category_map[row['ImageFullPath'].split('/')[0]]
            
            # Load và tiền xử lý ảnh
            image = load_img(image_path, target_size=(224, 224))
            image_array = img_to_array(image) / 255.0
            feature = model.predict(np.expand_dims(image_array, axis=0))
            features.append(feature.flatten())
            labels.append(label)
        except Exception as e:
            print(f"Error loading image {row['ImageFullPath']}: {e}")

    with ThreadPoolExecutor() as executor:
        executor.map(process_image, [row for _, row in df.iterrows()])

    # Lưu đặc trưng
    np.savez(features_file, features=np.array(features), labels=np.array(labels))
    print(f"Features saved to {features_file}")
    return np.array(features), np.array(labels)

In [6]:
# Đường dẫn đến file đặc trưng
features_file_train = os.path.join('/kaggle/input/split1-extract-features/Features_Train_Split_1.npz')
features_file_test = os.path.join('/kaggle/input/split1-extract-features/Features_Test_Split_1.npz')

# Tải dữ liệu
train_data = np.load(features_file_train)
X_train, y_train = train_data['features'], train_data['labels']

test_data = np.load(features_file_test)
X_test, y_test = test_data['features'], test_data['labels']

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Thiết lập SVM
classifier = SVC()

# Grid Search
param_grid = {
    'C': [0.1, 1, 10, 100],         # Regularization parameter
    'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],  # Kernel functions
    'degree': [2, 3, 4],            # Degree for polynomial kernel
    'gamma': [0.001, 0.01, 0.1, 1],  # Kernel coefficient for rbf, poly, sigmoid
}

random_search = RandomizedSearchCV(
    estimator=classifier,
    param_distributions=param_grid,
    n_iter=3,  # Số lượng tổ hợp thử nghiệm
    cv=2,
    scoring='accuracy',
    verbose=2,
    random_state=42 
)

random_search.fit(X_train, y_train)

# Best parameters
print(f"Best Parameters: {grid_search.best_params_}")

# Lấy mô hình với tham số tốt nhất
best_classifier = grid_search.best_estimator_

# Đánh giá trên tập test
accuracy = best_classifier.score(X_test, y_test)
y_pred = best_classifier.predict(X_test)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

# In kết quả
print(f"Final Accuracy: {accuracy:.4f}")
print(f"Confusion Matrix:\n{conf_matrix}")
print(f"Classification Report:\n{class_report}")

# Lưu kết quả
result_path = os.path.join(save_dir, "Final_Results_SVM_GridSearch.txt")
with open(result_path, 'w') as file:
    file.write(f"Final Accuracy: {accuracy:.4f}\n\n")
    file.write(f"Best Parameters: {grid_search.best_params_}\n\n")
    file.write(f"Confusion Matrix:\n{conf_matrix}\n\n")
    file.write(f"Classification Report:\n{class_report}")

# Lưu mô hình cuối cùng
model_path = os.path.join(save_dir, "Final_SVM_GridSearch.joblib")
joblib.dump(best_classifier, model_path)
print(f"Final model saved to {model_path}")